# Fetch TFRecords from Kaggle Dataset (Colab)

**Smart Sniper Spotter · Stage 2 · Transfer infrastructure (consumer side)**

Two cells you prepend to any Colab training notebook (NB2 rebuild, NB3 rebuild,
NB4 build) to make NB1's TFRecords available locally. Pulls ~10GB from the
private Kaggle Dataset published by the publishing notebook.

**Prereqs:**
- The publishing notebook ran successfully and the dataset exists at the
  slug `giladfaibish/smart-spotter-tfrecords-v1`
- Your `kaggle.json` file is accessible (you'll upload it via the Colab file
  picker on the first cell)

**Runtime per Colab session:** ~2 min download + ~30s unzip = ~3 min total.
Cached only for the current session; needs to re-run when the runtime resets.

**Where to place these cells in your training notebook:** **after** the env
setup cells (TF 2.13 install, OD API install, etc.) and **before** any cell
that reads TFRecord paths. The kaggle CLI install in Cell #B1 needs Python
to exist; the TFRecord path constants in Cell #B2 are consumed by the
pipeline.config writing cell.


## Cell #B1 — Install kaggle CLI and upload credentials


In [ ]:
# Cell #B1: Install kaggle CLI in the conda env and upload kaggle.json.
#
# We install the kaggle CLI into the *conda env* (which has TF 2.13) so that
# any TF code can read the downloaded TFRecords without crossing the
# kernel/subprocess boundary unnecessarily. The kaggle CLI itself is pure
# Python — install is fast (~10s).
#
# kaggle.json must contain {"username": "...", "key": "..."} — your Kaggle
# Legacy API credentials. Upload it via the Colab file picker; we'll move
# it to ~/.kaggle/ where the CLI looks.

import os
import json
from pathlib import Path

PIP = "/usr/local/bin/pip"
print(">> Installing kaggle CLI in conda env...")
!{PIP} install -q --no-cache-dir kaggle 2>&1 | tail -3

# Upload kaggle.json. Colab's files.upload() returns a dict of {filename: bytes}.
from google.colab import files
print("\n>> Please select your kaggle.json file when the file picker opens...")
uploaded = files.upload()

assert "kaggle.json" in uploaded, (
    "Expected a file named exactly 'kaggle.json'. Got: " + str(list(uploaded.keys())))

# Place it where the kaggle CLI expects it and set 0600 permissions
# (kaggle CLI refuses to read it otherwise).
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_bytes(uploaded["kaggle.json"])
kaggle_json.chmod(0o600)

# Verify the credentials parse cleanly.
creds = json.loads(kaggle_json.read_text())
assert "username" in creds and "key" in creds, \
    "kaggle.json must have 'username' and 'key' fields"
print(f"   ✓ Credentials installed for username: {creds['username']}")

# Verify the CLI can actually authenticate.
!kaggle --version
print("\n>> Listing your private datasets (auth check)...")
!kaggle datasets list --mine 2>&1 | head -10


>> Installing kaggle CLI in conda env...
object-detection 0.1 requires apache-beam, which is not installed.
tf-models-official 2.13.0 requires pyyaml<6.0,>=5.1, but you have pyyaml 6.0.3 which is incompatible.
object-detection 0.1 requires pyparsing==2.4.7, but you have pyparsing 3.3.2 which is incompatible.

>> Please select your kaggle.json file when the file picker opens...


Saving kaggle.json to kaggle.json
   ✓ Credentials installed for username: giladfaibish
Kaggle CLI 2.1.2

>> Listing your private datasets (auth check)...
Cannot specify both mine and a user


In [ ]:
!kaggle datasets list --mine 2>&1 | head -15

ref                                      title                                               size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------  -------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
giladfaibish/smart-spotter-tfrecords-v1  Smart Sniper Spotter — Stage 2 TFRecords v1  11206535924  2026-05-18 17:55:34.820000              0          0  0.5              


## Cell #B2 — Download dataset and resolve paths


In [ ]:
# Cell #B2: Download the TFRecord dataset and resolve paths for training.
#
# After this cell runs, the rest of the training notebook can use:
#   - TRAIN_PATTERN  → "/content/tfrecords/train-*.tfrecord*"
#   - VAL_PATTERN    → "/content/tfrecords/val-*.tfrecord*"
#   - LABEL_MAP_PATH → "/content/tfrecords/label_map.pbtxt"
#
# in pipeline.config and elsewhere.

import os
import glob
import subprocess

DATASET_SLUG = "giladfaibish/smart-spotter-tfrecords-v1"
DOWNLOAD_DIR = "/content/tfrecords_download"
TFRECORD_DIR = "/content/tfrecords"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(TFRECORD_DIR, exist_ok=True)

# `--unzip` extracts the dataset zip inline. `--force` re-downloads even if
# the zip is already in the cache, which we want when iterating in case the
# dataset got a new version.
print(f">> Downloading {DATASET_SLUG} (~10GB)...")
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --unzip 2>&1 | tail -20

# Kaggle's --unzip puts everything at the top level of DOWNLOAD_DIR.
# Move TFRecords + label_map to TFRECORD_DIR for clean paths.
import shutil

# TFRecords — could be under DOWNLOAD_DIR/tfrecords/ (matching publishing
# layout) or directly under DOWNLOAD_DIR (if Kaggle flattened the structure).
candidate_dirs = [
    os.path.join(DOWNLOAD_DIR, "tfrecords"),
    DOWNLOAD_DIR,
]
src_tfrecord_dir = None
for d in candidate_dirs:
    if glob.glob(os.path.join(d, "train-*.tfrecord*")):
        src_tfrecord_dir = d
        break
assert src_tfrecord_dir, (
    f"No train-*.tfrecord files found under {DOWNLOAD_DIR}. "
    f"Contents: {os.listdir(DOWNLOAD_DIR)}")

print(f"\n>> Found TFRecords in {src_tfrecord_dir}")
for src in glob.glob(os.path.join(src_tfrecord_dir, "*.tfrecord*")):
    dst = os.path.join(TFRECORD_DIR, os.path.basename(src))
    if src != dst:
        shutil.move(src, dst)

# label_map.pbtxt
label_map_candidates = (
    glob.glob(os.path.join(DOWNLOAD_DIR, "label_map.pbtxt")) +
    glob.glob(os.path.join(DOWNLOAD_DIR, "**/label_map.pbtxt"), recursive=True))
assert label_map_candidates, f"label_map.pbtxt not found in {DOWNLOAD_DIR}"
dst_label_map = os.path.join(TFRECORD_DIR, "label_map.pbtxt")
if label_map_candidates[0] != dst_label_map:
    shutil.copy2(label_map_candidates[0], dst_label_map)

# Optional metadata.
md_candidates = glob.glob(os.path.join(DOWNLOAD_DIR, "**/dataset_metadata.json"),
                          recursive=True)
if md_candidates:
    shutil.copy2(md_candidates[0], os.path.join(TFRECORD_DIR, "dataset_metadata.json"))

# Final resolution: paths the rest of the notebook will consume.
TRAIN_PATTERN = os.path.join(TFRECORD_DIR, "train-*.tfrecord*")
VAL_PATTERN = os.path.join(TFRECORD_DIR, "val-*.tfrecord*")
LABEL_MAP_PATH = os.path.join(TFRECORD_DIR, "label_map.pbtxt")

train_shards = sorted(glob.glob(TRAIN_PATTERN))
val_shards = sorted(glob.glob(VAL_PATTERN))
total_bytes = sum(os.path.getsize(p) for p in train_shards + val_shards)

print(f"\n>> TFRecord layout in {TFRECORD_DIR}:")
print(f"   Train shards: {len(train_shards)}")
print(f"   Val shards:   {len(val_shards)}")
print(f"   Total size:   {total_bytes/1e9:.2f} GB")
print(f"   Label map:    {LABEL_MAP_PATH} (exists={os.path.exists(LABEL_MAP_PATH)})")

assert len(train_shards) == 16, f"Expected 16 train shards, got {len(train_shards)}"
assert len(val_shards) == 4, f"Expected 4 val shards, got {len(val_shards)}"
assert os.path.exists(LABEL_MAP_PATH), "label_map.pbtxt is missing"

# Read-back sanity check via subprocess (TF lives in conda env, not the kernel).
print("\n>> Validating TFRecords by reading first example (subprocess)...")
validate_script = f"""
import glob, tensorflow as tf

for pattern, name in [({TRAIN_PATTERN!r}, 'train'), ({VAL_PATTERN!r}, 'val')]:
    paths = sorted(glob.glob(pattern))
    for raw in tf.data.TFRecordDataset(paths).take(1):
        ex = tf.train.Example()
        ex.ParseFromString(raw.numpy())
        feat = ex.features.feature
        fname = feat['image/filename'].bytes_list.value[0].decode()
        w = feat['image/width'].int64_list.value[0]
        h = feat['image/height'].int64_list.value[0]
        nb = len(feat['image/object/bbox/xmin'].float_list.value)
        print(f'{{name}} first example: {{fname}} {{w}}x{{h}} {{nb}} box(es)')
"""
result = subprocess.run(['/usr/local/bin/python', '-c', validate_script],
                        capture_output=True, text=True, timeout=120,
                        env={**os.environ, 'MPLBACKEND': 'Agg'})
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])
    raise RuntimeError("TFRecord validation failed.")

# Clean up the download dir to free disk space.
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
print(f"\n✓ TFRecords ready at {TFRECORD_DIR}")
print(f"   Constants exported: TRAIN_PATTERN, VAL_PATTERN, LABEL_MAP_PATH")


>> Downloading giladfaibish/smart-spotter-tfrecords-v1 (~10GB)...
Dataset URL: https://www.kaggle.com/datasets/giladfaibish/smart-spotter-tfrecords-v1
License(s): other
100%|██████████| 10.4G/10.4G [02:19<00:00, 80.3MB/s]


>> Found TFRecords in /content/tfrecords_download

>> TFRecord layout in /content/tfrecords:
   Train shards: 16
   Val shards:   4
   Total size:   11.30 GB
   Label map:    /content/tfrecords/label_map.pbtxt (exists=True)

>> Validating TFRecords by reading first example (subprocess)...
train first example: 000000135749.jpg 375x500 9 box(es)
val first example: 000000279073.jpg 640x425 4 box(es)


✓ TFRecords ready at /content/tfrecords
   Constants exported: TRAIN_PATTERN, VAL_PATTERN, LABEL_MAP_PATH
